# ✈️ NASA C-MAPSS: Comprehensive 4-Dataset Sensor & Regime Analysis

This notebook provides a complete physical, mathematical, and comparative analysis of all **four NASA C-MAPSS sub-datasets** (, , , and ).

### 📌 What This Notebook Covers:
1. **Dataset Architecture Matrix**: Comparison of operating conditions, fault modes, and engine trajectories.
2. **All 21 Sensors Physical Dictionary**: Engineering definitions, symbols, subsystems, physical units, and degradation characteristics.
3. **Sensor Variance Benchmark**: Exact empirical standard deviation and uniqueness scan across all 4 datasets to prove the **7 flat vs. 14 active** sensor phenomenon in / and full 21-sensor activity in /.
4. **Operating Regime Analysis (FD002 & FD004)**: 3D visualization of operational settings and implementation of **6-Regime Clustering & Condition-Based Normalization**.
5. **Fault Dynamics Comparison**: Single fault mode (HPC wear) vs. dual fault modes (HPC + Fan wear) in /.
6. **Preprocessing & Modeling Guidelines** for every sub-dataset.

## 📦 Step 0: Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Plotting style setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Column naming conventions per NASA documentation
index_cols = ['unit_nr', 'time_cycles']
setting_cols = ['setting_1', 'setting_2', 'setting_3']
sensor_cols = [f's_{i}' for i in range(1, 22)]
all_cols = index_cols + setting_cols + sensor_cols

print('Setup complete. Ready to load and inspect C-MAPSS datasets.')

## 📖 Step 1: All 21 Sensors — Physical Engineering Dictionary

Here is the complete engineering specification for every sensor variable in the commercial turbofan simulation:

| Sensor | Symbol | Description | Physical Unit | Baseline Mean (Sea Level) | Primary Degradation Trend |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **** | $ | Total temperature at fan inlet | \circ\text{R}$ | .67$ | **Constant at Sea Level** (Varies with altitude/speed) |
| **** | {24}$ | Total temperature at LPC outlet | \circ\text{R}$ | .68$ | **Upward Drift** (Compressor thermal loss) |
| **** | {30}$ | Total temperature at HPC outlet | \circ\text{R}$ | .52$ | **Upward Drift** (HPC efficiency degradation) |
| **** | {50}$ | Total temperature at LPT outlet | \circ\text{R}$ | .93$ | **Upward Drift** (Combustor & turbine thermal stress) |
| **** | $ | Pressure at fan inlet | $\text{psia}$ | .62$ | **Constant at Sea Level** (Ambient atmospheric pressure) |
| **** | {15}$ | Total pressure in bypass duct | $\text{psia}$ | .61$ | **Constant at Sea Level** (Duct pressure ratio locked) |
| **** | {30}$ | Total pressure at HPC outlet | $\text{psia}$ | .37$ | **Downward Drift** (Loss of compressor pressure buildup) |
| **** | $ | Physical fan speed | $\text{rpm}$ | .10$ | **Upward Drift** (Compensating for core thrust drop) |
| **** | $ | Physical core speed | $\text{rpm}$ | .24$ | **Upward Drift** (Spool works harder to maintain flow) |
| **** | $ | Engine pressure ratio ({50}/P_2$) | Dimensionless | .30$ | **Constant at Sea Level** (Commanded pressure setpoint) |
| **** | {30}$ | Static pressure at HPC outlet | $\text{psia}$ | .54$ | **Downward Drift** (Aerodynamic losses at diffuser) |
| **** | $\phi$ | Ratio of fuel flow to {30}$ | $\text{pps/psia}$ | .66$ | **Downward Drift** (More fuel needed per static pressure) |
| **** | {f\_Rcor}$ | Corrected fan speed | $\text{rpm}$ | .09$ | **Upward Drift** (Ambient-normalized low spool speed) |
| **** | {c\_Rcor}$ | Corrected core speed | $\text{rpm}$ | .75$ | **Upward Drift** (Ambient-normalized high spool speed) |
| **** | $ | Bypass ratio ({\text{fan}} / W_{\text{core}}$) | Dimensionless | .44$ | **Upward Drift** (Core flow restriction forces air to bypass) |
| **** | $ | Burner fuel-air ratio | Dimensionless | zsh.03$ | **Constant at Sea Level** (Stoichiometric control) |
| **** | $ | Bleed enthalpy | $\text{BTU/lbm}$ | .21$ | **Upward Drift** (Higher core discharge thermal energy) |
| **** | {f\_dmd}$ | Demanded fan speed | $\text{rpm}$ | .00$ | **Constant at Sea Level** (Control loop setpoint) |
| **** | \_dmd$ | Demanded corrected fan speed | $\%$ | .00$ | **Constant at Sea Level** (100% full rating setpoint) |
| **** | {31}$ | HPT coolant bleed | $\text{lbm/s}$ | .82$ | **Downward Drift** (Cooling airflow reduces over life) |
| **** | {32}$ | LPT coolant bleed | $\text{lbm/s}$ | .29$ | **Downward Drift** (Turbine cooling degradation) |

## 📊 Step 2: Automated Sensor Variance Scan Across All 4 Datasets

Let us load all 4 training files and compute the exact standard deviation ($\sigma$) and number of unique values for every sensor.

In [ ]:
datasets = ['FD001', 'FD002', 'FD003', 'FD004']
summary_records = []
data_dict = {}

for ds in datasets:
    df = pd.read_csv(f'train_{ds}.txt', sep=r'\s+', header=None, names=all_cols)
    data_dict[ds] = df
    
    n_engines = df['unit_nr'].nunique()
    total_rows = len(df)
    
    # Check standard deviations for all sensors
    for s in sensor_cols:
        std_val = df[s].std()
        nunique = df[s].nunique()
        is_flat = (std_val < 0.005) or (nunique <= 2)
        
        summary_records.append({
            'Dataset': ds,
            'Sensor': s,
            'StdDev': std_val,
            'UniqueCount': nunique,
            'Status': 'Flat / Constant' if is_flat else 'Active'
        })

sensor_summary_df = pd.DataFrame(summary_records)

# Pivot table to clearly compare status across datasets
status_pivot = sensor_summary_df.pivot(index='Sensor', columns='Dataset', values='Status')
display(status_pivot)

### 🔬 Count of Flat vs. Active Sensors by Dataset

In [ ]:
counts_df = sensor_summary_df.groupby(['Dataset', 'Status']).size().unstack(fill_value=0)
counts_df['Total Sensors'] = counts_df.sum(axis=1)

fig, ax = plt.subplots(figsize=(9, 4.5))
counts_df[['Active', 'Flat / Constant']].plot(kind='bar', stacked=True, color=['#2563eb', '#ef4444'], ax=ax, edgecolor='black')
plt.title('Sensor Activity Comparison across C-MAPSS Datasets', fontsize=14, fontweight='bold')
plt.xlabel('Sub-Dataset', fontsize=12)
plt.ylabel('Number of Sensors', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Sensor Status', loc='upper left')
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 0:
        ax.text(p.get_x() + width/2., p.get_y() + height/2., f'{int(height)}', ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

print('Sensor Counts Summary Table:')
display(counts_df)

## 🌐 Step 3: Operating Conditions & Flight Regimes (FD001 vs FD002)

Why are sensors flat in **FD001 & FD003**, but active in **FD002 & FD004**?

- In **FD001/FD003**, the engine is operated at a single sea-level condition.
- In **FD002/FD004**, the engine operates across **6 distinct flight regimes** defined by altitude (zsh\text{--}42,000\text{ ft}$), Mach number (zsh\text{--}0.84$), and throttle setting.

In [ ]:
fig = plt.figure(figsize=(15, 6))

# 3D plot for FD001 (1 condition)
ax1 = fig.add_subplot(121, projection='3d')
df_1 = data_dict['FD001'].sample(3000, random_state=42)
ax1.scatter(df_1['setting_1'], df_1['setting_2'], df_1['setting_3'], c='#2563eb', alpha=0.3, s=10)
ax1.set_title('FD001: 1 Single Operating Condition\n(Sea Level, Small Noise)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Setting 1 (Altitude)')
ax1.set_ylabel('Setting 2 (Mach Number)')
ax1.set_zlabel('Setting 3 (Throttle)')

# 3D plot for FD002 (6 regimes)
ax2 = fig.add_subplot(122, projection='3d')
df_2 = data_dict['FD002'].sample(3000, random_state=42)
ax2.scatter(df_2['setting_1'], df_2['setting_2'], df_2['setting_3'], c='#dc2626', alpha=0.3, s=10)
ax2.set_title('FD002: 6 Distinct Flight Regimes\n(Multi-Altitude & Variable Throttle)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Setting 1 (Altitude)')
ax2.set_ylabel('Setting 2 (Mach Number)')
ax2.set_zlabel('Setting 3 (Throttle)')

plt.tight_layout()
plt.show()

## 🛠️ Step 4: Regime-Conditioned Normalization for Multi-Condition Datasets (FD002 & FD004)

In **FD002** and **FD004**, normalizing with a global  destroys degradation patterns because regime variation is much larger than wear variation. 

Here is the standard industrial solution: **Cluster the 3 settings into 6 regimes with KMeans**, then normalize each sensor **within its respective regime**:

In [ ]:
def apply_regime_normalization(train_df, test_df, sensor_columns, n_regimes=6):
    """
    Clusters operating settings into 6 regimes using KMeans on train data,
    then normalizes sensor readings conditioned on the assigned regime.
    """
    train = train_df.copy()
    train[sensor_columns] = train[sensor_columns].astype(np.float32)
    test[sensor_columns] = test[sensor_columns].astype(np.float32)
    test = test_df.copy()
    
    # 1. Fit KMeans clusterer on operating settings
    kmeans = KMeans(n_clusters=n_regimes, random_state=42, n_init=10)
    train['regime'] = kmeans.fit_predict(train[setting_cols])
    test['regime'] = kmeans.predict(test[setting_cols])
    
    # 2. Conditioned Z-score scaling per regime
    scalers = {}
    for r in range(n_regimes):
        idx_train = (train['regime'] == r)
        idx_test = (test['regime'] == r)
        
        scaler = StandardScaler()
        if idx_train.sum() > 0:
            train.loc[idx_train, sensor_columns] = scaler.fit_transform(train.loc[idx_train, sensor_columns])
            scalers[r] = scaler
            if idx_test.sum() > 0:
                test.loc[idx_test, sensor_columns] = scaler.transform(test.loc[idx_test, sensor_columns])
                
    return train, test, scalers

# Load FD002 test to demonstrate
test_fd002 = pd.read_csv('test_FD002.txt', sep=r'\s+', header=None, names=all_cols)
train_norm_fd002, test_norm_fd002, scalers_fd002 = apply_regime_normalization(data_dict['FD002'], test_fd002, sensor_cols)

print(f'✅ Successfully normalized FD002 across {len(scalers_fd002)} flight regimes.')
print('Sample normalized readings for Engine #1 in FD002:')
display(train_norm_fd002[train_norm_fd002['unit_nr'] == 1][['unit_nr', 'time_cycles', 'regime', 's_2', 's_3', 's_4', 's_7']].head(6))

## 📈 Step 5: Visualizing Sensor Trajectories (FD001 vs FD002 vs FD003)

Let's compare the degradation curves of High-Pressure Compressor Outlet Temperature ({30}$ / ) and HPC Pressure ({30}$ / ) across datasets:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# FD001 Engine #1 (Single Condition, Single HPC Fault)
u1 = data_dict['FD001'][data_dict['FD001']['unit_nr'] == 1]
axes[0].plot(u1['time_cycles'], u1['s_3'], color='#2563eb', label='s_3 (HPC Temp T30)')
axes[0].set_title('FD001 (Single Condition: Clean Drift)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cycles')
axes[0].set_ylabel('s_3 Raw Temperature (°R)')
axes[0].legend()

# FD002 Engine #1 (Multi Condition - Raw vs Regime-Normalized)
u2_raw = data_dict['FD002'][data_dict['FD002']['unit_nr'] == 1]
u2_norm = train_norm_fd002[train_norm_fd002['unit_nr'] == 1]
axes[1].plot(u2_norm['time_cycles'], u2_norm['s_3'], color='#dc2626', label='s_3 (Regime Normalized)')
axes[1].set_title('FD002 (Multi-Regime Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cycles')
axes[1].set_ylabel('s_3 Normalized (Z-score)')
axes[1].legend()

# FD003 Engine #1 (Dual Fault: HPC + Fan)
u3 = data_dict['FD003'][data_dict['FD003']['unit_nr'] == 1]
axes[2].plot(u3['time_cycles'], u3['s_3'], color='#16a34a', label='s_3 (HPC Temp T30)')
axes[2].set_title('FD003 (Dual Fault Mode)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Cycles')
axes[2].set_ylabel('s_3 Raw Temperature (°R)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 📋 Step 6: Dataset Selection & Modeling Best Practices Cheat Sheet

| Dataset | Recommended Preprocessing | Sensor Feature Selection | Recommended Imputation | Recommended Prognostic Model |
| :--- | :--- | :--- | :--- | :--- |
| **** | Global  on active sensors | **Drop 7 flat sensors** (), keep 14 active. | MICE / KNN / Linear Interpolation | Random Forest / 1D-CNN / LSTM |
| **** | **KMeans 6-Regime Clustering** + per-regime scaling | **Keep all 21 sensors** (vital across flight altitudes & speeds). | Regime-aware KNN / MICE | Multi-layer LSTM / Temporal Convolutional Network (TCN) |
| **** | Global  on active sensors | **Drop 7 flat sensors**, keep 14 active + extract rolling volatility for fan wear. | MICE / KNN / Linear Interpolation | Attention-based Bi-LSTM / XGBoost |
| **** | **KMeans 6-Regime Clustering** + per-regime scaling | **Keep all 21 sensors** | Regime-aware MICE / Sequence Autoencoder | Transformer / Deep CNN-LSTM Ensemble |